# TF motif cooperativity within and across enhancers

Transcription factor motifs are located as *seqlets* in the ISM attributions of
each enhancer and annotated against JASPAR. Two perturbation experiments follow:

1. **Within enhancers:** seqlets of one enhancer are scrambled alone, in pairs and
   all together, and the effect on that enhancer's predicted accessibility is
   measured. Length-matched control windows give the specificity null.
2. **Across enhancers:** seqlets from all enhancers of a target gene are scrambled
   in one 1 Mb window, and interactions are estimated with the log-link GLM on
   target-gene expression and on the accessibility of every enhancer.

Interactions are deviations from a multiplicative null: negative is
super-additive, positive sub-additive.

**Inputs:** ISM attributions and stored predictions in `data/processed/`, the
JASPAR motifs in `data/meta/`, `data/raw/`, and the enhancer-level GLM tables
written by notebook 01.
**Outputs:** figures, `within_enhancer_TF_summary.txt` and
`cross_enhancer_summary.txt` in `results/summaries/`.

## Setup

In [ ]:
import ast
import itertools
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from alphagenome.data import gene_annotation, genome
from alphagenome.data import transcript as transcript_utils
from alphagenome.models import dna_client
from alphagenome.visualization import plot_components
from alphagenome_research.model.variant_scoring.gene_mask_extractor import (
    GeneMaskExtractor, GeneMaskType, GeneQueryType
)
from pyfaidx import Fasta
from tangermeme.annotate import annotate_seqlets
from tangermeme.ersatz import dinucleotide_shuffle
from tangermeme.io import read_meme
from tangermeme.plot import plot_logo
from tangermeme.seqlet import recursive_seqlets
from tangermeme.utils import one_hot_encode

# Make the package in ../src importable without installing it.
SRC_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import element_epistasis.GLM_fit as glmp
import element_epistasis.cross_level as cl
import element_epistasis.cross_plots as cp
import element_epistasis.cross_report as cr
import element_epistasis.enhancer_analysis as ea
import element_epistasis.glm as glm
import element_epistasis.tf_cooperativity as tc
import element_epistasis.tf_plots as tp
from element_epistasis import paths
from element_epistasis.ism import load_ism_dict
from element_epistasis.pipeline import run_all

paths.ensure_output_dirs()

### Inputs

ISM attributions come from `src/ism_attribution/`. The enhancer-level main effects
and interactions are written by notebook 01. Enhancer sequences are read from
hg38 when the FASTA is present; otherwise they are recovered from the ISM
tracks, whose scores sit on the reference base.

In [ ]:
# Enhancers and the published cooperativity TF list
def extract_genes(name_string):
    return [part.split('_')[0] for part in name_string.split(':')]
def extract_enhancers(name_string):
    return [part.split('_')[1] for part in name_string.split(':')]

enhancers = pd.read_csv(paths.RAW / "enhancers.bed", sep="\t", header=None,
                        names=["chrom", "start", "end", "name"])
enhancers['target_gene'] = enhancers['name'].apply(extract_genes)
enhancers['enhancer_nr'] = enhancers['name'].apply(extract_enhancers)
enhancers["ivs"] = [genome.Interval(chromosome=row.chrom, start=row.start, end=row.end)
                    for row in enhancers.itertuples()]

all_TFs = pd.read_csv(paths.RAW / "tfs.tsv", sep="\t", header=None, names=["TF"])
all_TFs = all_TFs.drop_duplicates().TF.to_list()

# ISM attributions, one entry per enhancer in alphabetical order (defines seq_idx)
ism_enhancer = load_ism_dict(paths.PROCESSED / "K562_ISM__enhancers.h5")

# Enhancer sequences
if paths.HG38_FASTA.exists():
    hg38 = Fasta(str(paths.HG38_FASTA))
    enhancers["seq"] = [str(hg38[e.chrom][e.start:e.end]) for e in enhancers.itertuples()]
else:
    hg38 = None
    _nuc = np.array(list("ACGT"))
    enhancers["seq"] = [
        "".join(_nuc[tc.reference_onehot_from_ism(ism_enhancer[n]["ATAC"]).argmax(axis=0)])
        for n in enhancers["name"]]

# GENCODE v46: MANE transcripts for ISM plots, protein-coding genes for windows and exon masks
gtf = paths.load_gtf()
transcript_extractor = transcript_utils.TranscriptExtractor(gene_annotation.filter_to_mane_select_transcript(gtf))
gtf_df = gtf[gtf.gene_type == "protein_coding"]

# JASPAR 2026 vertebrate motifs
meme = read_meme(paths.JASPAR_MEME)
meme_names = list(meme.keys())

# Enhancer-level GLM results from notebook 01, and the measured CRISPRi pairs
enh_mains = pd.read_csv(paths.PROCESSED / "enhancer_insilico_single_KO_main_effects.csv")
enh_inters = pd.read_csv(paths.PROCESSED / "enhancer_insilico_KO_inters.csv")
measured_pairs = cl.normalise_measured_pairs(pd.read_csv(paths.RAW / "pair_interaction_adj.csv"), collapse="mean")

enhancers

In [ ]:
# Seqlet calling and annotation
SEQLET_MIN_LEN = 6                       # shortest seqlet (bp)
SEQLET_MAX_LEN = 12                      # longest seqlet (bp)
SEQLET_THRESHOLD = 0.05                  # recursive_seqlets significance threshold
MOTIF_N_NEAREST = 10                     # JASPAR matches returned per seqlet
MOTIF_MATCH_P = 0.001                    # p-value threshold for a motif match
SEQLET_TRIM_CUTOFF = 0.7                 # trim edges below this fraction of the mean per-base attribution
MIN_SEQLET_ATTRIBUTION = 2               # minimum summed ATAC attribution of a seqlet

# Within-enhancer scramble design
CONTEXT_LENGTH = dna_client.SEQUENCE_LENGTH_1MB   # AlphaGenome input window, centred on the enhancer
ENHANCER_LENGTH = 313                    # every enhancer in the BED file is 313 bp
MAX_SEQLETS_WITHIN_RUN = 5               # seqlets kept per enhancer (strongest, non-overlapping)
WITHIN_DESIGN = "reduced"                # baseline, singles, pairs and all-scrambled
N_SHUFFLES = 10                          # dinucleotide shuffles per scrambled combination
SHUFFLE_SEED = 42                        # seed of the dinucleotide shuffles
N_CONTROLS = 3                           # control windows per enhancer
CONTROL_ATTRIBUTION_PERCENTILE = 50.0    # controls drawn from windows at or below this attribution percentile
API_CHUNK_SIZE = 64                      # sequences per AlphaGenome request
API_MAX_WORKERS = 6                      # parallel AlphaGenome requests

# Cross-enhancer design
MAX_SEQLETS_PER_ENHANCER = 5             # seqlets per enhancer in the gene-level design
MAX_SEQLETS_PER_GENE = 20                # seqlets per gene in the gene-level design
TOP_K_MATCHES = 5                        # motif matches checked against the TF list for eligibility
INCLUDE_WITHIN_ENHANCER_PAIRS = True     # also measure within-enhancer pairs in the gene window
INPUT_BLOCK = 128                        # 1 Mb inputs built at once (bounds memory, ~1 MB each)
GLM_FAMILY = "gamma"                     # response distribution of the interaction GLM
MAX_INTERACTION_ORDER = 2                # highest interaction order of the GLM

# Examples shown in figures
EXAMPLE_ISM_ENHANCERS = ["ALDH1A1_enh1", "ALDH1A1_enh2:ANXA1_enh1", "SLC25A37_enh1"]
EXAMPLE_SEQLET_TRACKS = [(0, "ATAC"), (0, "RNA ALDH1A1"), (0, "TF STAT5A"),
                         (68, "ATAC"), (68, "TF STAT5A"), (50, "ATAC")]   # (seq_idx, ISM track)
EXAMPLE_CROSS_LOCI = {"SLC25A37": ["SLC25A37_enh1", "SLC25A37_enh2"],
                      "CXXC5": ["CXXC5_enh1", "CXXC5_enh2"],
                      "FAM89A": ["FAM89A_enh1", "FAM89A_enh3"]}         # gene: enhancers whose ATAC fit is shown

## ISM attributions

In silico mutagenesis scores every possible substitution in an enhancer; the
attribution at a base is the predicted change at the reference nucleotide.
Tracks: accessibility (ATAC), target-gene RNA and K562 TF ChIP-seq.

In [ ]:
def plot_ISM(res_dict, interval, variants = None, title = ""):
    seq_logo_data = [plot_components.SeqLogo(
                            scores=attributions,
                            scores_interval=interval,
                            ylabel= f"{m}",
                        ) for m, attributions in res_dict.items()
                    ]

    if variants:
        annot = [plot_components.VariantAnnotation(variants, alpha=0.15, colors="yellow", label_angle=50),
                    # plot_components.VariantAnnotation(sig_mutations, alpha=0.5, colors="red")
                    ]
    else: annot = None
    plot_components.plot(
        [plot_components.TranscriptAnnotation(transcripts = transcript_extractor.extract(interval)), *seq_logo_data],
        #seq_logo_data,
        title = f"{title}",
        interval=interval,
        fig_width=35,
        annotations=annot,
    )
    plt.show()


for name in EXAMPLE_ISM_ENHANCERS:
    plot_ISM(ism_enhancer[name], enhancers[enhancers.name == name].ivs.values[0])

## Seqlet calling

Seqlets are short stretches of high ATAC attribution, called recursively on the
per-base attribution summed over the four nucleotides. The reference sequence
is recovered from the ISM matrices for motif annotation.

In [ ]:
# Per-base ATAC attribution, one row per enhancer (row index = seq_idx)
att, names = [], []
for name, ism_res in ism_enhancer.items():
    att.append(np.array(ism_res["ATAC"].sum(axis=1)))
    names.append(name)
atac_att = torch.tensor(np.stack(att))

# Reference one-hot from the ISM matrices
atac_stacked = torch.tensor(np.stack([ism_res["ATAC"] for name, ism_res in ism_enhancer.items()]))
indices = torch.argmax(torch.abs(atac_stacked), dim=-1)
onehot = torch.nn.functional.one_hot(indices, num_classes=4).permute(0, 2, 1)

seqlets = recursive_seqlets(atac_att, min_seqlet_len=SEQLET_MIN_LEN,
                            max_seqlet_len=SEQLET_MAX_LEN, threshold=SEQLET_THRESHOLD)
seqlets

## Motif annotation

Each seqlet is compared with every JASPAR motif; matches below the p-value
threshold are kept, ranked by significance.

In [ ]:
m_idx, m_p = annotate_seqlets(onehot, seqlets, meme, n_nearest=MOTIF_N_NEAREST)

collapsed_p = pd.Series([tensor.item() for sublist in m_p for tensor in sublist])
plt.plot(-np.log10(collapsed_p.sort_values(ascending=False).values))
plt.xlabel("Hit (Sorted by pval)")
plt.ylabel("-log10(pval)")
plt.hlines(y=-np.log10(MOTIF_MATCH_P), xmin=0, xmax=len(collapsed_p))
plt.title("Pval per Hit")
plt.show()

In [ ]:
# Significant matches per seqlet, as factor names
motif_names, m_p_kept = [], []
for motif_idxs, pvals in zip(m_idx, m_p):
    meme_idx, meme_pvals = [], []
    for i, p in zip(motif_idxs, pvals):
        if p.item() <= MOTIF_MATCH_P:
            meme_idx.append(meme_names[i.item()].upper().split(" ")[1])
            meme_pvals.append(p.item())
    motif_names.append(meme_idx)
    m_p_kept.append(meme_pvals)

seqlets["seq_idx"] = seqlets["example_idx"]
seqlets["example_idx"] = [str([name for name in names]) for names in motif_names]
seqlets["annot_p_val"] = [str([p for p in ps]) for ps in m_p_kept]
seqlets

## Seqlet trimming and TF labels

Seqlet edges with little attribution are trimmed. Each seqlet is labelled with
its best match, preferring factors on the cooperativity TF list; the full ranked
match list is kept as an alternative annotation. Seqlets without any match are
dropped.

In [ ]:
def trim_seqlets(seqlets, attributions, cutoff=0.8):
    '''Function to trim seqlet edges so that a base explains at least the cutoff percentage of the average attribution for that base'''
    rows = []
    for seq in seqlets.itertuples(index=False):
        s = seq.start
        e = seq.end
        l = e - s
        seq_id = seq.seq_idx
        sum_a = seq.attribution
        avg_a_per_pos = sum_a / l
        
        seq_attributions = attributions[seq_id, s:e]
        base_contributing = torch.where(seq_attributions / avg_a_per_pos > cutoff)[0]        
        row_data = seq._asdict()
        
        # Safety check ensure there is at least six bases meet the cutoff
        if len(base_contributing) > 5:
            first_true = base_contributing[0].item()
            last_true = base_contributing[-1].item() + 1 
            new_s = s + first_true
            new_e = s + last_true
            row_data['start'] = new_s
            row_data['end'] = new_e
               
            row_data['attribution'] = attributions[seq_id, new_s:new_e].sum().item()           
            rows.append(row_data)
        else:
            rows.append(row_data) #pass the og seqlet
            
    trimmed_seqlets = pd.DataFrame(rows)    
    return trimmed_seqlets

trimmed_seqlets = trim_seqlets(seqlets, atac_att, cutoff=SEQLET_TRIM_CUTOFF)

In [ ]:
def simplify_tf_name(raw_name: str, families: set) -> str:
    raw_name_lower = raw_name.upper()
    
    # Exact match
    if raw_name_lower in families:
        return raw_name_lower
    # Partial match: the longest listed name contained in the motif name, ties
    # going to the one written first (DDIT3::CEBPA -> DDIT3). Looping over the
    # set directly left ties to string hashing, which changes between sessions.
    hits = [family for family in families if family in raw_name_lower]
    if hits:
        return min(hits, key=lambda family: (-len(family), raw_name_lower.find(family)))
            
    # No match
    return raw_name

def tidy_tf_names(df, tf_names, tf_col="example_idx", pval_col="annot_p_val"):
    tf_names_lower = {name.upper() for name in tf_names}

    def parse(val):
        if isinstance(val, list): return val
        try: return ast.literal_eval(val)
        except: return []

    selected_tfs, selected_pvals = [], []

    for _, row in df.iterrows():
        tfs = parse(row[tf_col])
        pvals = parse(row[pval_col])
 
        if not tfs: #halde case were no motif was found in jaspar aka empty list
            selected_tfs.append("Not found")
            selected_pvals.append(float("nan"))
        elif len(tfs) == 0: #halde case were no motif was found in jaspar aka empty list
            selected_tfs.append("Not found")
            selected_pvals.append(float("nan"))
        else:
            #check for matches
            matches = [(i, simplify_tf_name(tf, tf_names_lower)) for i, tf in enumerate(tfs) if simplify_tf_name(tf, tf_names_lower) in tf_names_lower]
            if len(matches) == 0:
                selected_tfs.append(tfs[0])
                selected_pvals.append(pvals[0])
            else:
                selected_tfs.append(matches[0][1])
                selected_pvals.append(pvals[0])

    return (
        pd.Series(selected_tfs, index=df.index, name="tf_name"),
        pd.Series(selected_pvals, index=df.index, name="pval"),
    )

In [ ]:
trimmed_seqlets["example_idx_alterntive"], trimmed_seqlets["annot_p_val_alterntive"] = trimmed_seqlets["example_idx"].copy(), trimmed_seqlets["annot_p_val"].copy()
trimmed_seqlets["example_idx"], trimmed_seqlets["annot_p_val"] = tidy_tf_names(trimmed_seqlets, all_TFs)

trimmed_seqlets = trimmed_seqlets[trimmed_seqlets.example_idx != "Not found"]
trimmed_seqlets

### Attribution threshold

Number of enhancers keeping one, two, three or more seqlets as the attribution
cutoff rises; seqlets below the chosen cutoff are removed.

In [ ]:
plt.plot(trimmed_seqlets.attribution.sort_values(ascending=True).values)
plt.xlabel("Hit (Sorted by Attribution)")
plt.ylabel("Attribution")
plt.hlines(y=3, xmin = 0, xmax=len(trimmed_seqlets))
plt.title("Sum Attribution per Hit")
plt.show()

In [ ]:
df = trimmed_seqlets
max_co = 10
cutoffs = np.arange(0, max_co, 0.25)
 
counts = {1: [], 2: [], 3: [], "4+": []}
 
for cutoff in cutoffs:
    passing = df[df["attribution"] > cutoff]
    per_seq = passing.groupby("seq_idx").size()
 
    counts[1].append((per_seq == 1).sum())
    counts[2].append((per_seq == 2).sum())
    counts[3].append((per_seq == 3).sum())
    counts["4+"].append((per_seq >= 4).sum())
 
fig, ax = plt.subplots(figsize=(12, 5))
 
colors = {"1 seqlet": "#185FA5", "2 seqlets": "#1D9E75", "3 seqlets": "#BA7517", "4+ seqlets": "black"}
labels = ["1 seqlet", "2 seqlets", "3 seqlets", "4+ seqlets"]
data_series = [counts[1], counts[2], counts[3], counts["4+"]]
 
bottom = np.zeros(len(cutoffs))
for label, series, color in zip(labels, data_series, colors.values()):
    bars = np.array(series)
    ax.bar(cutoffs, bars, bottom=bottom, width=0.25, label=label, color=color)
    bottom += bars
 
ax.set_xlabel("Attribution cutoff", fontsize=12)
ax.set_ylabel("Number of sequences with Seqlet", fontsize=12)
ax.set_title("Sequences with seqlets above attribution cutoff", fontsize=13)
ax.set_xticks(np.arange(0, max_co+1, 0.5))
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
 
plt.tight_layout()
plt.savefig(paths.FIGURES / "seqlet_stacked_bar.png", dpi=150)
plt.show()

In [ ]:
trimmed_seqlets = trimmed_seqlets[trimmed_seqlets.attribution >= MIN_SEQLET_ATTRIBUTION]

### Seqlets on the attribution logos

In [ ]:
for seq_idx, track in EXAMPLE_SEQLET_TRACKS:
    print(names[seq_idx], track)
    fig, ax = plt.subplots(1, 1, figsize=(20, 4))
    ism_res = torch.tensor(ism_enhancer[names[seq_idx]][track]).permute(1, 0)
    plot_logo(ism_res, ax=ax,
              annotations=trimmed_seqlets.loc[trimmed_seqlets.seq_idx == seq_idx, ["example_idx_alterntive", "start", "end", "attribution"]],
              score_key='attribution', n_tracks=5)
    plt.tight_layout()
    plt.show()

In [ ]:
ylim = atac_att.min().item(), atac_att.max().item()

df = trimmed_seqlets
multi_seq_idx = df.seq_idx.value_counts()[df.seq_idx.value_counts() > 1].index
df = df[df.seq_idx.isin(multi_seq_idx)]

fig, axes = plt.subplots(df.seq_idx.nunique(), 1, figsize = (20, 50))
print(len(axes))

for ax, seq_i in enumerate(np.sort(df.seq_idx.unique())):
    name = names[seq_i]
    ism_res = torch.tensor(ism_enhancer[name]["ATAC"]).permute(1, 0)
    plot_logo(ism_res, ax=axes[ax], annotations=df[df.seq_idx == seq_i], score_key='attribution', n_tracks=5)
    axes[ax].set_ylim(ylim)
    axes[ax].set_ylabel(f"{name} {seq_i}", rotation=0)

plt.tight_layout()
plt.show()    

## Within-enhancer scramble design

Repeated labels within an enhancer are numbered, and each enhancer keeps its
strongest non-overlapping seqlets. Every enhancer gets dinucleotide-shuffled
copies that donate the scrambled bases. Control windows are matched in length
but drawn from positions without called seqlets.

In [ ]:
trimmed_seqlets["name"] = trimmed_seqlets.seq_idx.map(dict(zip(range(0, len(names), 1), names)))
fulls_seqlets = pd.merge(trimmed_seqlets, enhancers, how="left", on="name", suffixes=("_seq", ""))

fulls_seqlets = tc.make_seqlet_labels_unique(
    fulls_seqlets, enhancer_col="name", label_col="example_idx", start_col="start_seq"
)
fulls_seqlets = tc.cap_seqlets_per_enhancer(
    fulls_seqlets, max_seqlets=MAX_SEQLETS_WITHIN_RUN, enhancer_col="name",
    score_col="attribution", start_col="start_seq", end_col="end_seq",
)
print(tc.summarise_design_budget(fulls_seqlets, n_shuffles=N_SHUFFLES, design=WITHIN_DESIGN).tail(1))

seq_1hot = torch.stack([one_hot_encode(seq) for seq in enhancers.seq])
dishuff = dinucleotide_shuffle(seq_1hot, n=N_SHUFFLES, random_state=SHUFFLE_SEED)
dishuff.shape

In [ ]:
control_windows = tc.sample_control_windows(
    exclude_seqlets=fulls_seqlets,
    attributions=atac_att,
    enhancer_names=names,
    n_controls=N_CONTROLS,
    attribution_percentile=CONTROL_ATTRIBUTION_PERCENTILE,
)
control_windows = control_windows.merge(
    enhancers[["name", "chrom", "start", "end", "seq", "target_gene"]],
    on="name", how="left",
)
print(tc.summarise_design_budget(control_windows, design="singles").tail(1))

## Within-enhancer predictions

The cells tagged `prediction` in this section rerun AlphaGenome for every seqlet
and control combination. They need an API key (`ALPHAGENOME_API_KEY` or
`~/.alphagenome_api_key`), the hg38 FASTA in `data/reference/` and several
hours of API time.

**To skip them**, run only the next cell, which loads the stored predictions,
and continue at the consistency check.

In [ ]:
# Skip the regeneration: load the stored predictions
scram_effects_raw = pd.read_parquet(paths.PROCESSED / "TF_insilico_KO_raw.parquet", engine='pyarrow')
ctrl_effects_raw = pd.read_parquet(paths.PROCESSED / "TF_insilico_KO_controls_raw.parquet", engine='pyarrow')
scram_effects_raw.head()

### AlphaGenome model and K562 tracks

Shared by the within- and cross-enhancer prediction steps.

In [ ]:
model = dna_client.create(paths.alphagenome_api_key())

track_metadata = model.output_metadata(dna_client.Organism.HOMO_SAPIENS).concatenate()
K562_metadata = track_metadata[(track_metadata["biosample_name"].str.contains("K562", case=False, na=False)) &
                               ((track_metadata["output_type"] == dna_client.OutputType.ATAC) |
                                (track_metadata["output_type"] == dna_client.OutputType.CHIP_TF))
                               ].dropna(how="all", axis=1)
K562_curie = K562_metadata.ontology_curie.unique().tolist()
print("curie:", K562_curie)

exon_extractor = GeneMaskExtractor(
    gtf=gtf_df,
    gene_mask_type=GeneMaskType.EXONS,
    gene_query_type=GeneQueryType.INTERVAL_CONTAINED,
    filter_protein_coding=False
)

### Combinations and model inputs

Each combination replaces the scrambled seqlets with the same positions of a
shuffled copy of the enhancer, once per shuffle; the modified enhancer is then
placed in the centre of its 1 Mb genomic window.

In [ ]:
# ACGT order assumed (standard one-hot for genomics)
_IDX_TO_NUC = np.array(["A", "C", "G", "T"])


def onehot_to_seq(onehot: torch.Tensor) -> str:
    """
    Convert a one-hot tensor of shape [4, L] to an uppercase nucleotide string.
    Handles ties by taking argmax (first max index).
    """
    indices = onehot.argmax(dim=0).numpy()   # shape [L]
    return "".join(_IDX_TO_NUC[indices])


def splice_scramble(
    original_seq: str,
    scramble_windows: list[tuple[int, int]],
    donor_seq: str,
) -> str:
    """
    Replace every (start, end) window in `original_seq` with the same window
    from `donor_seq`.  start/end are 0-based, end exclusive (Python slicing).
    """
    seq = list(original_seq)
    for start, end in scramble_windows:
        seq[start:end] = list(donor_seq[start:end])
    return "".join(seq)


def generate_combination_df(
    motif_df: pd.DataFrame,
    dishuff: torch.Tensor,
    enhancer_col: str = "name",
    seq_col: str = "seq",
    seq_idx_col: str = "seq_idx",
    start_col: str = "start_seq",
    end_col: str = "end_seq",
    design: str = "reduced",
    seqlet_name_col: str = "example_idx",   # human-readable seqlet label
) -> pd.DataFrame:
    """
    Parameters
    ----------
    motif_df    : DataFrame as described (one row per seqlet)
    dishuff     : Tensor [n_enhancers, 10, 4, seq_len], dinucleotide shuffles
    *_col       : column name overrides

    Returns
    -------
    pd.DataFrame with columns:
        name, seq_idx, combo_id, combo_label,
        scrambled_seqlets, unscrambled_seqlets,
        n_scrambled, shuffle_idx, sequence
    """
    n_shuffles = dishuff.shape[1]
    rows = []

    for enhancer, grp in motif_df.groupby(enhancer_col):
        grp = grp.reset_index(drop=True)

        original_seq = grp[seq_col].iloc[0].upper()
        seq_idx      = int(grp[seq_idx_col].iloc[0])

        # Seqlets for this enhancer, sorted by start position
        grp_sorted = grp.sort_values(start_col).reset_index(drop=True)
        n_seqlets  = len(grp_sorted)

        # Pre-convert all 10 shuffled tensors to strings once
        enhancer_shuffles = dishuff[seq_idx]          # [10, 4, L]
        shuffle_seqs = [
            onehot_to_seq(enhancer_shuffles[i])
            for i in range(n_shuffles)
        ]

        # Seqlet metadata lists (aligned with grp_sorted rows)
        seqlet_labels  = grp_sorted[seqlet_name_col].tolist()
        seqlet_windows = list(
            zip(grp_sorted[start_col].tolist(), grp_sorted[end_col].tolist())
        )

        # Combinations of the chosen design (reduced: baseline, singles, pairs, all)
        for combo_id, combo in enumerate(tc.enumerate_combinations(n_seqlets, design)):
            scrambled_idx   = list(combo)
            unscrambled_idx = [i for i in range(n_seqlets) if i not in combo]

            scrambled_labels   = [seqlet_labels[i]  for i in scrambled_idx]
            unscrambled_labels = [seqlet_labels[i]  for i in unscrambled_idx]
            scrambled_windows  = [seqlet_windows[i] for i in scrambled_idx]

            # Human-readable label, e.g. "scr[GATA2,STAT5A]_unscr[MYB73]"
            combo_label = (
                f"scr[{','.join(str(s) for s in scrambled_labels)}]"
                f"_unscr[{','.join(str(s) for s in unscrambled_labels)}]"
            )

            if combo_id == 0:
                # ── all unscrambled: use original sequence, no shuffle ──
                rows.append(
                    dict(
                        name       = enhancer,
                        seq_idx             = seq_idx,
                        combo_id            = combo_id,
                        combo_label         = combo_label,
                        scrambled_seqlets   = scrambled_labels,
                        unscrambled_seqlets = unscrambled_labels,
                        n_scrambled         = 0,
                        shuffle_idx         = None,
                        sequence            = original_seq,
                    )
                )
            else:
                # ── scrambled combo: one row per shuffle ──
                for shuf_i, donor in enumerate(shuffle_seqs):
                    scrambled_seq = splice_scramble(
                        original_seq, scrambled_windows, donor
                    )
                    rows.append(
                        dict(
                            name       = enhancer,
                            seq_idx             = seq_idx,
                            combo_id            = combo_id,
                            combo_label         = combo_label,
                            scrambled_seqlets   = scrambled_labels,
                            unscrambled_seqlets = unscrambled_labels,
                            n_scrambled         = len(scrambled_idx),
                            shuffle_idx         = shuf_i,
                            sequence            = scrambled_seq,
                        )
                    )

    result = pd.DataFrame(rows)

    # Tidy column order
    col_order = [
        "name", "seq_idx", "combo_id", "combo_label",
        "n_scrambled", "scrambled_seqlets", "unscrambled_seqlets",
        "shuffle_idx", "sequence",
    ]
    return result[col_order]




def make_model_inputs(
    combo_df: pd.DataFrame,
    hg38: Fasta,
    input_length: int = 2 ** 20,
    chrom_col: str = "chrom",
    start_col: str = "start",
    end_col: str = "end",
    seq_col: str = "sequence",
) -> list[str]:
    """
    Parameters
    ----------
    combo_df      : output of generate_combination_df(), one row per sequence
    hg38          : pyfaidx.Fasta handle to the hg38 genome
    input_length  : total length of the model input (e.g. 2**19 = 524_288)
    *_col         : column name overrides

    Returns
    -------
    List of uppercase strings of length `input_length`, in the same row order
    as combo_df.
    """
    results: list[str] = []

    for _, row in combo_df.iterrows():
        chrom     = row[chrom_col]
        enh_start = int(row[start_col])   # 0-based, BED-style
        enh_end   = int(row[end_col])     # 0-based exclusive
        enh_seq   = row[seq_col].upper()  # our generated sequence

        enh_len   = enh_end - enh_start
        assert len(enh_seq) == enh_len, (
            f"Generated sequence length {len(enh_seq)} != "
            f"enhancer span {enh_len} for {chrom}:{enh_start}-{enh_end}"
        )

        # ── center the enhancer in the input window ──────────────────────────
        enh_center = (enh_start + enh_end) // 2

        win_start  = enh_center - input_length // 2
        win_end    = win_start  + input_length          # always exactly input_length

        # ── fetch genomic flanks, N-padding at chromosome boundaries ─────────
        chrom_len  = len(hg38[chrom])

        # How much to clip on each side
        left_pad  = max(0, -win_start)          # bp to N-pad on the left
        right_pad = max(0, win_end - chrom_len) # bp to N-pad on the right

        fetch_start = max(0,          win_start)
        fetch_end   = min(chrom_len,  win_end)

        genomic = hg38[chrom][fetch_start:fetch_end].seq.upper()

        # Rebuild with N-pads if near chromosome boundary
        if left_pad or right_pad:
            genomic = "N" * left_pad + genomic + "N" * right_pad

        assert len(genomic) == input_length, (
            f"Fetched {len(genomic)} bp, expected {input_length} "
            f"(chrom={chrom}, win={win_start}-{win_end})"
        )

        # ── splice generated sequence into window ─────────────────────────────
        # Position of enhancer relative to window start
        enh_in_win_start = enh_start - win_start + left_pad
        enh_in_win_end   = enh_in_win_start + enh_len

        full_seq = (
            genomic[:enh_in_win_start]
            + enh_seq
            + genomic[enh_in_win_end:]
        )

        assert len(full_seq) == input_length, (
            f"Output length {len(full_seq)} != {input_length}"
        )

        results.append(full_seq)

    return results

In [ ]:
control_df = generate_combination_df(control_windows, dishuff, design="singles")
control_df = pd.merge(control_df, enhancers[["chrom", "start", "end", "name", "target_gene"]],
                      on="name", how="left")

out_df = generate_combination_df(fulls_seqlets, dishuff, design=WITHIN_DESIGN)
out_df = pd.merge(out_df, enhancers[["chrom", "start", "end", "name", "target_gene"]], how="left", on="name")

print(f"seqlet sequences: {len(out_df)}   control sequences: {len(control_df)}")
seq_lens = out_df.groupby("name")["sequence"].apply(lambda s: s.str.len().unique())
assert all(len(v) == 1 for v in seq_lens), "Sequence length mismatch!"

### Predict accessibility and expression

Accessibility is the predicted ATAC signal summed over the enhancer; expression
is the mean predicted RNA coverage over the exons of each target gene.

In [ ]:
def mean_of_lists(series):
    """Element-wise mean across a series of lists."""
    parsed = [
        ast.literal_eval(v) if isinstance(v, str) else v
        for v in series
    ]
    return np.array(parsed).mean(axis=0).tolist()

def predict_seqlet_KO_effects(combinations_df, genome = hg38, seq_len = CONTEXT_LENGTH, model = model, enhancers = enhancers, curies = K562_curie):
    df_list = []
    df_list_raw = []
    number_enhs = len(combinations_df.name.unique())
    for enh_nr, name in enumerate(combinations_df.name.unique()):
        print(f"Analysing enhancer: {name} ({enh_nr}/{number_enhs})")
        enh_scram = combinations_df[combinations_df.name == name]

        model_inputs = make_model_inputs(enh_scram, genome, input_length=seq_len)

        # Sanity: verify the enhancer region is our sequence, not raw genome
        row0       = enh_scram.iloc[0]
        enh_center = (row0["start"] + row0["end"]) // 2
        win_start  = enh_center - seq_len // 2
        offset     = row0["start"] - win_start
        embedded   = model_inputs[0][offset : offset + (row0["end"] - row0["start"])]
        assert embedded == row0["sequence"].upper(), "Sanity check failed!"
        print("Sanity check passed.")

        enhancer_length =  (enhancers[enhancers.name == name].end.values - enhancers[enhancers.name == name].start.values)[0]
        input_interval = enhancers[enhancers.name == name].ivs.values[0].resize(seq_len)


        out_pred = tc.predict_with_backoff(
            model=model,
            sequences=model_inputs,
            interval=input_interval,
            requested_outputs=[dna_client.OutputType["ATAC"], dna_client.OutputType["RNA_SEQ"]],
            ontology_terms=curies,
            chunk_size=API_CHUNK_SIZE,
            max_workers=API_MAX_WORKERS,
        )

        #get gene masks once for enhancer
        gene_mask, annot = exon_extractor.extract(interval = out_pred[0].rna_seq.interval)
        is_target = annot.gene_name.isin(enhancers[enhancers.name == name].target_gene.values[0])
        annot = annot.loc[is_target]
        gene_mask = gene_mask[:, is_target]

        #ATAC aggreagarion mask
        agg_start = (seq_len - enhancer_length) // 2
        agg_end = agg_start + enhancer_length


        #iterate ove rshuffels
        target_genes, RNA_counts, ATAC_counts = [], [], []
        for seq_out in out_pred:

            #summ aatac coutns
            ATAC_counts.append(seq_out.atac.values.mean(axis = 1)[agg_start:agg_end].sum())

            #sum gene counts       
            targets, counts = [], []
            for mask_idx, gene in enumerate(annot.itertuples()):
                strand = gene.strand
                stranded_track = seq_out.rna_seq.metadata.strand.isin([strand, "."])
                gene_counts = seq_out.rna_seq.values[:,stranded_track].mean(axis = 1)[gene_mask[:, mask_idx]].mean()

                targets.append(gene.gene_name)
                counts.append(gene_counts)
            target_genes.append(targets)
            RNA_counts.append(counts)

        del out_pred
        enh_scram["ATAC_counts"], enh_scram["RNA_counts"], enh_scram["target_gene"], =  ATAC_counts, RNA_counts, target_genes

        #aggregate the shuffeld ref sequnces by avg
        df_list_raw.append(enh_scram.copy())
        enh_scram_agg = (
            enh_scram
            .groupby(
                ["name", "seq_idx", "combo_label", "n_scrambled", "chrom", "start", "end"],
                observed=True,
            )
            .agg(
                ATAC_counts         = ("ATAC_counts",         "mean"),
                RNA_counts          = ("RNA_counts",           mean_of_lists),
                target_gene         = ("target_gene",          "first"),
                scrambled_seqlets   = ("scrambled_seqlets",    "first"),
                unscrambled_seqlets = ("unscrambled_seqlets",  "first"),
            )
            .reset_index()
        )

        df_list.append(enh_scram_agg)
        del enh_scram, enh_scram_agg
        
    return df_list_raw, df_list

In [ ]:
control_list_raw, control_list = predict_seqlet_KO_effects(control_df)
ctrl_effects_raw = pd.concat(control_list_raw).reset_index(drop=True)
ctrl_effects = pd.concat(control_list).reset_index(drop=True)
ctrl_effects_raw.to_parquet(paths.PROCESSED / "TF_insilico_KO_controls_raw.parquet", engine='pyarrow', index=False)
ctrl_effects.to_parquet(paths.PROCESSED / "TF_insilico_KO_controls.parquet", engine='pyarrow', index=False)

In [ ]:
TF_seqlet_list_raw, TF_seqlet_list = predict_seqlet_KO_effects(out_df)
scram_effects_raw = pd.concat(TF_seqlet_list_raw).reset_index(drop=True)
scram_effects = pd.concat(TF_seqlet_list).reset_index(drop=True)
scram_effects_raw.to_parquet(paths.PROCESSED / "TF_insilico_KO_raw.parquet", engine='pyarrow', index=False)
scram_effects.to_parquet(paths.PROCESSED / "TF_insilico_KO.parquet", engine='pyarrow', index=False)

### Consistency check

Seqlets are recomputed from the ISM file on every run, so the stored predictions
must use the same seqlet labels and enhancer order.

In [ ]:
_stored = {lab for labs in scram_effects_raw["scrambled_seqlets"] for lab in labs}
_missing = _stored - set(fulls_seqlets["example_idx"])
assert not _missing, f"stored seqlet labels not reproduced: {sorted(_missing)[:10]}"
_idx = scram_effects_raw.drop_duplicates("name").set_index("name")["seq_idx"]
assert all(names[i] == n for n, i in _idx.items()), "seq_idx order differs from the ISM file"

## Within-enhancer motif effects and interactions

Single-motif effects, pairwise interactions and per-enhancer cooperativity are
computed by direct contrast on the log scale, pairing replicates that share the
same shuffle. The supplementary figure holds the specificity control and the
checks for floor and saturation artefacts.

In [ ]:
motifs    = tc.add_bh_correction(tc.build_motif_effects(scram_effects_raw, "ATAC_counts", seqlets=fulls_seqlets))
pairs     = tc.add_bh_correction(tc.build_pairwise_interactions(scram_effects_raw, "ATAC_counts", seqlets=fulls_seqlets))
coop_atac = tc.add_bh_correction(tc.build_cooperativity_table(scram_effects_raw, "ATAC_counts"))
coop_rna  = tc.add_bh_correction(tc.build_cooperativity_table(scram_effects_raw, "RNA_counts"))

controls  = tc.build_motif_effects(ctrl_effects_raw, "ATAC_counts", seqlets=control_windows)

In [ ]:
fig = tp.make_main_figure(coop_atac, coop_rna, pairs, motifs, controls=controls, figsize=(10, 12))
plt.show()

In [ ]:
split = tc.build_saturation_split(scram_effects_raw, value_col="ATAC_counts")
floor_df = tc.build_floor_check(scram_effects_raw, value_col="ATAC_counts")
dose = tc.build_dose_response(scram_effects_raw, value_col="ATAC_counts", seqlets=fulls_seqlets)

fig = tp.make_supplementary_figure(motifs, coop_atac, pairs, controls=controls,
                                   split=split, floor_df=floor_df, dose=dose,
                                   figsize=(10, 14))
plt.show()

Closely spaced motifs also have the largest summed main effects, so the
dependence of the interaction on spacing is re-tested with the summed main
effects partialled out.

In [ ]:
spacing = cl.summarise_spacing_partial(
    pairs, spacing_col="spacing_bp", value_col="interaction",
    delta_a_col="delta_a", delta_b_col="delta_b",
)
spacing

### Text summary

All within-enhancer statistics in `results/summaries/within_enhancer_TF_summary.txt`.

In [ ]:
tables = run_all(
    tf_scram=scram_effects_raw,
    tf_seqlets=fulls_seqlets,
    tf_control_scram=ctrl_effects_raw,
    tf_control_windows=control_windows,
    top_n=20,
    summary_path=paths.SUMMARIES / "within_enhancer_TF_summary.txt",
)

## Cross-enhancer scramble design

All enhancers of a target gene are perturbed together in one window that
contains the gene body and every enhancer. Seqlet labels carry their enhancer
(`GENE_enhN|MOTIF`), so each pair is classified as *within* or *across*
enhancers, and both kinds are measured in the same window.

### Gene-level enhancers and windows

An element assigned to two genes is perturbed separately in each gene's window.
Genes with a single enhancer cannot form cross-enhancer pairs and are dropped.

In [ ]:
assert dishuff.shape[1] >= N_SHUFFLES, "dishuff has fewer shuffles than requested"
assert dishuff.shape[-1] == ENHANCER_LENGTH, f"dishuff sequence length is not {ENHANCER_LENGTH}"

In [ ]:
# seq_idx must come from the ISM key order, because that is what indexes dishuff.
name_to_seq_idx = {name: i for i, name in enumerate(names)}

gene_enh_rows = []
for _, row in enhancers.iterrows():
    # Each part of the compound name is one "GENE_enhN" label.
    for part in str(row["name"]).split(":"):
        gene = part.split("_")[0]
        gene_enh_rows.append(dict(
            gene=gene,
            enh_key=part,                 # gene-qualified, unique project-wide
            bed_name=row["name"],         # the ISM key, may be compound
            seq_idx=name_to_seq_idx.get(row["name"], np.nan),
            chrom=row["chrom"],
            start=int(row["start"]),
            end=int(row["end"]),
            seq=row["seq"].upper(),
        ))

gene_enhancers = pd.DataFrame(gene_enh_rows)

# A missing seq_idx means the enhancer has no ISM and therefore no seqlets.
missing = gene_enhancers["seq_idx"].isna()
if missing.any():
    print(f"[warn] {int(missing.sum())} enhancer(s) absent from the ISM set, dropped:")
    print(gene_enhancers.loc[missing, 'enh_key'].tolist())
    gene_enhancers = gene_enhancers[~missing].copy()
gene_enhancers["seq_idx"] = gene_enhancers["seq_idx"].astype(int)

print(f"genes={gene_enhancers.gene.nunique()}  "
      f"gene-enhancer pairs={len(gene_enhancers)}")
print(gene_enhancers.groupby("gene").size().describe()[["min", "50%", "max"]])

In [ ]:
# ASSUMPTION: gtf_df carries gene_name, seqname/chromosome, start and end.
# Column naming differs between GENCODE feather releases, so resolve it once.
_chrom_col = next(c for c in ["Chromosome", "seqname", "chrom", "contig"]
                  if c in gtf_df.columns)
gene_bodies = (gtf_df[gtf_df["gene_name"].isin(gene_enhancers["gene"].unique())]
               .groupby("gene_name")
               .agg(gene_chrom=(_chrom_col, "first"),
                    gene_start=("Start" if "Start" in gtf_df.columns else "start", "min"),
                    gene_end=("End" if "End" in gtf_df.columns else "end", "max")))

window_rows = []
for gene, group in gene_enhancers.groupby("gene"):
    chrom = group["chrom"].iloc[0]
    lo, hi = int(group["start"].min()), int(group["end"].max())

    # Extend to cover the gene body when the annotation is available. If the
    # gene is not in the GTF the enhancer span alone is used, which still
    # centres the window sensibly but is worth knowing about.
    if gene in gene_bodies.index:
        body = gene_bodies.loc[gene]
        if str(body["gene_chrom"]).replace("chr", "") == str(chrom).replace("chr", ""):
            lo = min(lo, int(body["gene_start"]))
            hi = max(hi, int(body["gene_end"]))
    else:
        print(f"[warn] {gene} not found in GTF; window built from enhancers only")

    centre = (lo + hi) // 2
    win_start = centre - CONTEXT_LENGTH // 2
    window_rows.append(dict(gene=gene, chrom=chrom, span_bp=hi - lo,
                            win_start=win_start, win_end=win_start + CONTEXT_LENGTH,
                            n_enhancers=len(group)))

gene_windows = pd.DataFrame(window_rows).set_index("gene")

# Everything must fit with margin; a truncated enhancer would silently corrupt
# the design rather than raise.
too_wide = gene_windows[gene_windows["span_bp"] > CONTEXT_LENGTH - 2 * ENHANCER_LENGTH]
assert too_wide.empty, f"gene span exceeds the window:\n{too_wide}"

# Genes with one enhancer cannot contribute a cross-enhancer pair.
single = gene_windows[gene_windows["n_enhancers"] < 2]
if not single.empty:
    print(f"[info] dropping {len(single)} single-enhancer gene(s): "
          f"{single.index.tolist()}")
    gene_windows = gene_windows.drop(single.index)
    gene_enhancers = gene_enhancers[gene_enhancers.gene.isin(gene_windows.index)]

print(f"genes retained: {len(gene_windows)}")
print(f"span (kb): median={gene_windows.span_bp.median()/1000:.0f}  "
      f"max={gene_windows.span_bp.max()/1000:.0f}")

### TF eligibility and seqlet selection

A seqlet is eligible if any of its top motif matches belongs to the cooperativity
TF list. Within a gene, enhancers are taken in order of their enhancer-level
effect on expression; within an enhancer, eligible seqlets come first, then
higher attribution, under per-enhancer and per-gene caps.

In [ ]:
coop_tf_set = {str(t).strip().upper() for t in all_TFs if str(t).strip()}
print(f"cooperativity TF list: {len(coop_tf_set)} unique symbols")

def _eligible_from_matches(match_list) -> bool:
    # True if any of the top K matches simplifies to a listed factor.
    if not isinstance(match_list, (list, tuple)):
        return False
    for raw in list(match_list)[:TOP_K_MATCHES]:
        if simplify_tf_name(str(raw), coop_tf_set) in coop_tf_set:
            return True
    return False

seqlets_annot = fulls_seqlets.copy()
seqlets_annot["example_idx_alterntive"] = seqlets_annot["example_idx_alterntive"].apply(ast.literal_eval)
seqlets_annot["top_matches"] = seqlets_annot["example_idx_alterntive"].apply(
    lambda v: list(v)[:TOP_K_MATCHES] if isinstance(v, (list, tuple)) else [])
seqlets_annot["tf_eligible"] = seqlets_annot["top_matches"].apply(_eligible_from_matches)

print(f"eligible seqlets (top {TOP_K_MATCHES} matches): {int(seqlets_annot.tf_eligible.sum())}/"
      f"{len(seqlets_annot)} ({seqlets_annot.tf_eligible.mean():.1%})")

In [ ]:
# Join seqlets onto the exploded gene table. A shared enhancer produces one
# seqlet row per gene, which is intended.
seqlets_gene = seqlets_annot.merge(
    gene_enhancers[["gene", "enh_key", "bed_name", "seq_idx", "chrom",
                    "start", "end", "seq"]],
    left_on="name", right_on="bed_name", how="inner", suffixes=("", "_enh"),
)

# Enhancer-local -> genomic.
seqlets_gene["genomic_start"] = seqlets_gene["start"] + seqlets_gene["start_seq"]
seqlets_gene["genomic_end"] = seqlets_gene["start"] + seqlets_gene["end_seq"]
seqlets_gene["label"] = (seqlets_gene["enh_key"] + "|"
                         + seqlets_gene["example_idx"].astype(str))

# Every seqlet must sit inside its parent enhancer.
assert (seqlets_gene["start_seq"] >= 0).all(), "negative seqlet start"
assert (seqlets_gene["end_seq"] <= ENHANCER_LENGTH).all(), "seqlet runs past enhancer end"
assert (seqlets_gene["genomic_start"] >= seqlets_gene["start"]).all()
assert (seqlets_gene["genomic_end"] <= seqlets_gene["end"]).all()

# Labels must be unique within a gene, otherwise the design silently merges cells.
dupes = seqlets_gene.groupby(["gene", "label"]).size()
assert (dupes <= 1).all(), f"duplicate labels within a gene:\n{dupes[dupes > 1]}"

print(f"seqlets mapped: {len(seqlets_gene)} across "
      f"{seqlets_gene.gene.nunique()} genes")

In [ ]:
# ASSUMPTION: `enh_mains` is the enhancer-level GLM main effect table with
# `full_a` holding "GENE_enhN" and `coef` holding beta_i on RNA.
beta_lookup = (enh_mains.drop_duplicates("full_a")
                        .set_index("full_a")["coef"].abs())

selection = seqlets_gene.copy()
selection["enh_beta_abs"] = selection["enh_key"].map(beta_lookup)

# Enhancers absent from the GLM table get rank last rather than being dropped.
n_missing_beta = int(selection["enh_beta_abs"].isna().sum())
if n_missing_beta:
    print(f"[warn] {n_missing_beta} seqlet(s) on enhancers with no beta_i; "
          f"those enhancers are ranked last")
selection["enh_beta_abs"] = selection["enh_beta_abs"].fillna(-np.inf)

kept = []
for gene, group in selection.groupby("gene"):
    # Enhancers strongest first.
    enh_order = (group.groupby("enh_key")["enh_beta_abs"].first()
                      .sort_values(ascending=False).index.tolist())

    chosen_for_gene = []
    for enh_key in enh_order:
        block = group[group["enh_key"] == enh_key]
        # Eligible first, then by attribution. Non-overlap is enforced greedily,
        # since overlapping windows cannot be perturbed independently.
        block = block.sort_values(["tf_eligible", "attribution"],
                                  ascending=[False, False])
        accepted = []
        for _, row in block.iterrows():
            if len(accepted) >= MAX_SEQLETS_PER_ENHANCER:
                break
            if any(row["start_seq"] < a["end_seq"] and a["start_seq"] < row["end_seq"]
                   for a in accepted):
                continue
            accepted.append(row)
        chosen_for_gene.extend(accepted)

        if len(chosen_for_gene) >= MAX_SEQLETS_PER_GENE:
            break

    kept.extend(chosen_for_gene[:MAX_SEQLETS_PER_GENE])

selected_seqlets = pd.DataFrame(kept).reset_index(drop=True)

# Report what the rules actually did.
per_gene = selected_seqlets.groupby("gene").agg(
    n_seqlets=("label", "size"),
    n_enhancers=("enh_key", "nunique"),
    n_eligible=("tf_eligible", "sum"),
)
per_gene["gene_cap_bound"] = per_gene["n_seqlets"] >= MAX_SEQLETS_PER_GENE
print(f"selected {len(selected_seqlets)} seqlets across {len(per_gene)} genes")
print(f"seqlets per gene: median={per_gene.n_seqlets.median():.0f}  "
      f"range=[{per_gene.n_seqlets.min()},{per_gene.n_seqlets.max()}]")
print(f"genes where the per-gene cap bound: {int(per_gene.gene_cap_bound.sum())}")
print(f"eligible fraction among selected: "
      f"{selected_seqlets.tf_eligible.mean():.1%}")

# Genes that lost an enhancer entirely contribute fewer cross-enhancer pairs.
lost = per_gene[per_gene.n_enhancers < gene_windows.loc[per_gene.index, "n_enhancers"]]
if not lost.empty:
    print(f"[info] {len(lost)} gene(s) have enhancers with no selected seqlet:")
    print(lost.head(10).to_string())

### Design size

Baseline, every single seqlet, every pair and the all-scrambled state, which
keeps the cost quadratic rather than exponential in the number of seqlets.

In [ ]:
def enumerate_gene_combinations(labels: list[str],
                                include_within: bool = True) -> list[tuple]:
    """Baseline, singletons, pairs, and the all-scrambled state.

    Restricting to pairs keeps the cost O(N^2) rather than O(2^N). Pairs are
    classified downstream as within-enhancer or across-enhancer by comparing
    the "GENE_enhN" prefix of the two labels.

    Args:
        labels: Seqlet labels for one gene.
        include_within: Keep pairs whose members share an enhancer. Setting
            this False saves budget but removes the controlled comparison
            between within-enhancer and cross-enhancer interaction.

    Returns:
        Sorted list of label tuples, always starting with the empty tuple.
    """
    combos = [()]
    combos += [(label,) for label in labels]
    for a, b in itertools.combinations(labels, 2):
        same_enhancer = a.split("|")[0] == b.split("|")[0]
        if same_enhancer and not include_within:
            continue
        combos.append((a, b))
    if len(labels) > 2:
        combos.append(tuple(labels))
    return sorted(set(combos), key=lambda c: (len(c), c))


budget_rows = []
for gene, group in selected_seqlets.groupby("gene"):
    labels = group["label"].tolist()
    combos = enumerate_gene_combinations(labels, INCLUDE_WITHIN_ENHANCER_PAIRS)
    n_within = sum(1 for c in combos
                   if len(c) == 2 and c[0].split("|")[0] == c[1].split("|")[0])
    n_across = sum(1 for c in combos
                   if len(c) == 2 and c[0].split("|")[0] != c[1].split("|")[0])
    budget_rows.append(dict(
        gene=gene, n_seqlets=len(labels), n_combos=len(combos),
        n_within_pairs=n_within, n_across_pairs=n_across,
        # One deterministic baseline, everything else replicated.
        n_sequences=1 + (len(combos) - 1) * N_SHUFFLES,
    ))

budget = pd.DataFrame(budget_rows)
print(budget.to_string(index=False))
print(f"\nTOTAL sequences: {budget.n_sequences.sum():,}")
print(f"  across-enhancer pairs: {budget.n_across_pairs.sum():,}")
print(f"  within-enhancer pairs: {budget.n_within_pairs.sum():,}")

## Cross-enhancer predictions

The cells tagged `prediction` rerun AlphaGenome for every gene-level combination.
They need the model cell from *Within-enhancer predictions*, the API key, the
hg38 FASTA and several hours of API time.

**To skip them**, run only the next cell, which loads the stored predictions,
and continue at the consistency check.

In [ ]:
# Skip the regeneration: load the stored predictions
cross_scram_effects_raw = pd.read_parquet(paths.PROCESSED / "TF_cross_enhancer_raw.parquet")
cross_scram_effects = pd.read_parquet(paths.PROCESSED / "TF_cross_enhancer.parquet")
cross_scram_effects_raw.head()

### Gene-level combinations

Each combination modifies every enhancer of the gene; the same shuffle index is
used for all enhancers of one replicate, so replicates stay paired across
enhancers.

In [ ]:
_IDX_TO_NUC = np.array(["A", "C", "G", "T"])


def _onehot_to_seq(onehot: torch.Tensor) -> str:
    """Convert a [4, L] one-hot tensor to an uppercase nucleotide string."""
    return "".join(_IDX_TO_NUC[onehot.argmax(dim=0).numpy()])


def generate_gene_combination_df(
    seqlets: pd.DataFrame,
    gene_enhancers: pd.DataFrame,
    dishuff: torch.Tensor,
    n_shuffles: int = N_SHUFFLES,
    include_within: bool = INCLUDE_WITHIN_ENHANCER_PAIRS,
) -> pd.DataFrame:
    """Enumerate every scramble combination per gene.

    Args:
        seqlets: Selected seqlets with `gene`, `enh_key`, `label`, `seq_idx`,
            `start_seq`, `end_seq`.
        gene_enhancers: Gene-qualified enhancer table with reference sequences.
        dishuff: Donor shuffles, shape (n_enhancers, n_shuffles, 4, 313).
        n_shuffles: Replicates per non-trivial combination.
        include_within: Whether to retain within-enhancer pairs.

    Returns:
        One row per sequence, carrying the modified sequence of every enhancer
        of the gene as an aligned list.
    """
    rows = []

    for gene, group in seqlets.groupby("gene"):
        # Enhancers of this gene, in a fixed order. This order defines the
        # alignment of `enh_sequences`, `ATAC_counts` and `target_enhancer`.
        enh_table = (gene_enhancers[gene_enhancers.gene == gene]
                     .drop_duplicates("enh_key")
                     .sort_values("start")
                     .reset_index(drop=True))
        enh_keys = enh_table["enh_key"].tolist()
        reference = {r.enh_key: r.seq for r in enh_table.itertuples()}
        seq_idx_of = {r.enh_key: int(r.seq_idx) for r in enh_table.itertuples()}

        # Pre-render the donor shuffles once per enhancer.
        donors = {
            key: [_onehot_to_seq(dishuff[seq_idx_of[key]][k])
                  for k in range(n_shuffles)]
            for key in enh_keys
        }

        # Windows to splice, keyed by label.
        windows = {r.label: (r.enh_key, int(r.start_seq), int(r.end_seq))
                   for r in group.itertuples()}

        combos = enumerate_gene_combinations(list(windows), include_within)

        for combo_id, combo in enumerate(combos):
            # The unperturbed reference is deterministic and needs no replicates.
            shuffle_indices = [None] if not combo else range(n_shuffles)

            for k in shuffle_indices:
                sequences = dict(reference)   # copy per row
                for label in combo:
                    enh_key, start, end = windows[label]
                    donor = donors[enh_key][k]
                    seq = list(sequences[enh_key])
                    seq[start:end] = list(donor[start:end])
                    sequences[enh_key] = "".join(seq)

                enhancers_hit = {label.split("|")[0] for label in combo}
                if len(combo) == 2:
                    pair_type = "within" if len(enhancers_hit) == 1 else "across"
                elif len(combo) == 1:
                    pair_type = "single"
                elif len(combo) == 0:
                    pair_type = "baseline"
                else:
                    pair_type = "all"

                rows.append(dict(
                    name=gene,                       # matches enhancer-level schema
                    combo_id=combo_id,
                    combo_label="scr[" + ",".join(combo) + "]",
                    scrambled_seqlets=list(combo),
                    unscrambled_seqlets=[l for l in windows if l not in combo],
                    n_scrambled=len(combo),
                    n_enhancers_hit=len(enhancers_hit),
                    pair_type=pair_type,
                    shuffle_idx=np.nan if k is None else k,
                    enh_keys=enh_keys,
                    enh_sequences=[sequences[key] for key in enh_keys],
                ))

    return pd.DataFrame(rows)


cross_df = generate_gene_combination_df(selected_seqlets, gene_enhancers, dishuff)

print(f"rows: {len(cross_df):,}   genes: {cross_df.name.nunique()}")
print(cross_df.pair_type.value_counts().to_string())

# Every modified sequence must still be exactly one enhancer long.
lengths = {len(s) for row in cross_df.enh_sequences for s in row}
assert lengths == {ENHANCER_LENGTH}, f"unexpected sequence lengths: {lengths}"

# The baseline row must be identical to the reference for every enhancer.
for gene, group in cross_df.groupby("name"):
    base = group[group.n_scrambled == 0].iloc[0]
    ref = (gene_enhancers[gene_enhancers.gene == gene]
           .drop_duplicates("enh_key").sort_values("start"))
    assert base["enh_sequences"] == ref["seq"].tolist(), \
        f"baseline sequence altered for {gene}"
print("baseline rows match the reference sequences.")

### Model inputs and predictions

The 1 Mb reference window is fetched once per gene and the modified enhancers are
spliced in at their offsets. Readouts: RNA of the target gene and ATAC summed over
each enhancer of the gene.

In [ ]:
def build_gene_window(gene: str, hg38, gene_windows: pd.DataFrame) -> str:
    """Fetch the reference 1 Mb window for a gene, N-padded at contig ends."""
    info = gene_windows.loc[gene]
    chrom = info["chrom"]
    win_start, win_end = int(info["win_start"]), int(info["win_end"])

    chrom_len = len(hg38[chrom])
    left_pad = max(0, -win_start)
    right_pad = max(0, win_end - chrom_len)
    fetch_start = max(0, win_start)
    fetch_end = min(chrom_len, win_end)

    core = str(hg38[chrom][fetch_start:fetch_end]).upper()
    window = "N" * left_pad + core + "N" * right_pad
    assert len(window) == CONTEXT_LENGTH, f"window length {len(window)} != {CONTEXT_LENGTH}"
    return window


def enhancer_offsets(gene: str, enh_keys: list[str],
                     gene_enhancers: pd.DataFrame,
                     gene_windows: pd.DataFrame) -> list[int]:
    """Offset of each enhancer inside the gene window, in the given order."""
    win_start = int(gene_windows.loc[gene, "win_start"])
    lookup = (gene_enhancers[gene_enhancers.gene == gene]
              .drop_duplicates("enh_key").set_index("enh_key")["start"])
    offsets = [int(lookup[key]) - win_start for key in enh_keys]
    # An enhancer falling outside the window would be silently truncated.
    for key, off in zip(enh_keys, offsets):
        assert 0 <= off and off + ENHANCER_LENGTH <= CONTEXT_LENGTH, \
            f"{key} lies outside the window for {gene} (offset {off})"
    return offsets


def make_gene_model_inputs(block: pd.DataFrame, window: str,
                           offsets: list[int]) -> list[str]:
    """Splice each row's modified enhancers into a copy of the gene window."""
    inputs = []
    for _, row in block.iterrows():
        seq = list(window)
        for offset, enh_seq in zip(offsets, row["enh_sequences"]):
            seq[offset:offset + ENHANCER_LENGTH] = list(enh_seq)
        inputs.append("".join(seq))
    return inputs

In [ ]:
def predict_cross_enhancer_effects(
    combinations_df: pd.DataFrame,
    gene_enhancers: pd.DataFrame,
    gene_windows: pd.DataFrame,
    hg38,
    model,
    curies,
    seq_len: int = CONTEXT_LENGTH,
    input_block: int = INPUT_BLOCK,
):
    """Predict RNA and per-enhancer ATAC for every combination, gene by gene."""
    raw_frames, agg_frames = [], []
    genes = combinations_df["name"].unique()

    for gene_nr, gene in enumerate(genes, start=1):
        block_all = combinations_df[combinations_df.name == gene].copy()
        enh_keys = block_all["enh_keys"].iloc[0]
        print(f"[{gene_nr}/{len(genes)}] {gene}: {len(block_all)} sequences, "
              f"{len(enh_keys)} enhancers", flush=True)

        window = build_gene_window(gene, hg38, gene_windows)
        offsets = enhancer_offsets(gene, enh_keys, gene_enhancers, gene_windows)

        # Sanity: the baseline input must reproduce the reference genome at
        # every enhancer offset. If this fails the coordinate mapping is wrong.
        baseline_row = block_all[block_all.n_scrambled == 0].iloc[[0]]
        check_input = make_gene_model_inputs(baseline_row, window, offsets)[0]
        for key, offset, ref_seq in zip(enh_keys, offsets,
                                        baseline_row["enh_sequences"].iloc[0]):
            assert check_input[offset:offset + ENHANCER_LENGTH] == ref_seq, \
                f"embedding mismatch for {key} in {gene}"

        interval = genome.Interval(
            chromosome=str(gene_windows.loc[gene, "chrom"]),
            start=int(gene_windows.loc[gene, "win_start"]),
            end=int(gene_windows.loc[gene, "win_end"]),
        )

        gene_mask = annot = None
        atac_all, rna_all, target_all = [], [], []

        for begin in range(0, len(block_all), input_block):
            block = block_all.iloc[begin:begin + input_block]
            inputs = make_gene_model_inputs(block, window, offsets)

            out_pred = tc.predict_with_backoff(
                model=model,
                sequences=inputs,
                interval=interval,
                requested_outputs=[dna_client.OutputType["ATAC"],
                                   dna_client.OutputType["RNA_SEQ"]],
                ontology_terms=curies,
                chunk_size=API_CHUNK_SIZE,
                max_workers=API_MAX_WORKERS,
            )
            del inputs

            # Gene masks depend only on the interval, so extract them once.
            if gene_mask is None:
                first = next(p for p in out_pred if p is not None)
                gene_mask, annot = exon_extractor.extract(
                    interval=first.rna_seq.interval)
                is_target = annot.gene_name == gene
                annot = annot.loc[is_target]
                gene_mask = gene_mask[:, is_target.values]
                if annot.empty:
                    print(f"  [warn] {gene} has no exon mask in this window; "
                          f"RNA will be empty")

            for seq_out in out_pred:
                if seq_out is None:
                    # A failed chunk. Recorded as missing rather than dropped,
                    # so row alignment with block_all is preserved.
                    atac_all.append([np.nan] * len(enh_keys))
                    rna_all.append([np.nan])
                    target_all.append([gene])
                    continue

                atac_track = seq_out.atac.values.mean(axis=1)
                atac_all.append([float(atac_track[off:off + ENHANCER_LENGTH].sum())
                                 for off in offsets])

                counts, targets = [], []
                for mask_idx, row in enumerate(annot.itertuples()):
                    stranded = seq_out.rna_seq.metadata.strand.isin(
                        [row.strand, "."])
                    value = (seq_out.rna_seq.values[:, stranded].mean(axis=1)
                             [gene_mask[:, mask_idx]].mean())
                    counts.append(float(value))
                    targets.append(row.gene_name)
                rna_all.append(counts if counts else [np.nan])
                target_all.append(targets if targets else [gene])

            del out_pred

        block_all["ATAC_counts"] = atac_all
        block_all["RNA_counts"] = rna_all
        block_all["target_gene"] = target_all
        block_all["target_enhancer"] = [enh_keys] * len(block_all)

        # The heavy sequence columns are not needed downstream.
        block_all = block_all.drop(columns=["enh_sequences"])
        raw_frames.append(block_all.copy())

        # Aggregate over shuffles, mirroring the within-enhancer pipeline.
        agg = (block_all
               .groupby(["name", "combo_label", "n_scrambled", "pair_type"],
                        observed=True)
               .agg(ATAC_counts=("ATAC_counts",
                                 lambda s: np.nanmean(np.vstack(s), axis=0).tolist()),
                    RNA_counts=("RNA_counts",
                                lambda s: np.nanmean(np.vstack(s), axis=0).tolist()),
                    scrambled_seqlets=("scrambled_seqlets", "first"),
                    unscrambled_seqlets=("unscrambled_seqlets", "first"),
                    target_enhancer=("target_enhancer", "first"),
                    target_gene=("target_gene", "first"),
                    n_enhancers_hit=("n_enhancers_hit", "first"))
               .reset_index())
        agg_frames.append(agg)

    return raw_frames, agg_frames

In [ ]:
raw_frames, agg_frames = predict_cross_enhancer_effects(
    cross_df, gene_enhancers, gene_windows, hg38, model, K562_curie)

cross_scram_effects_raw = pd.concat(raw_frames).reset_index(drop=True)
cross_scram_effects = pd.concat(agg_frames).reset_index(drop=True)
cross_scram_effects_raw.to_parquet(paths.PROCESSED / "TF_cross_enhancer_raw.parquet")
cross_scram_effects.to_parquet(paths.PROCESSED / "TF_cross_enhancer.parquet")

print(f"raw: {cross_scram_effects_raw.shape}   agg: {cross_scram_effects.shape}")
print(f"missing predictions: "
      f"{int(cross_scram_effects_raw.ATAC_counts.apply(lambda v: np.isnan(v).any()).sum())}"
      f" of {len(cross_scram_effects_raw)} rows")

# ATAC values must align with the enhancer list of each row
bad = cross_scram_effects_raw.apply(lambda r: len(r["ATAC_counts"]) != len(r["target_enhancer"]), axis=1)
assert not bad.any(), f"{int(bad.sum())} rows have misaligned ATAC vectors"

### Consistency check

The seqlet selection is recomputed, so the stored predictions must use the same
gene-level seqlet labels.

In [ ]:
_stored = {lab for labs in cross_scram_effects_raw["scrambled_seqlets"] for lab in labs}
_missing = _stored - set(selected_seqlets["label"])
assert not _missing, f"stored cross-enhancer labels not reproduced: {sorted(_missing)[:10]}"

## Cross-enhancer interaction GLM

The log-link GLM of notebook 01 is fitted with seqlets as the perturbation unit:
once per gene on target-gene expression and once per enhancer locus on
accessibility. Interaction terms are labelled within or across enhancer, and
across-enhancer motif interactions are aggregated to the enhancer pair they span.

In [ ]:
# One RNA value per row (the window's target gene)
cross_scram_effects_raw["RNA_counts"] = [l[0] for l in cross_scram_effects_raw.RNA_counts]

# Expression
cross_glm_res_rna = glm.fit_all_genes(
    df_raw=cross_scram_effects_raw,
    name_col="name",
    response_col="RNA_counts",
    family=GLM_FAMILY,
    max_order=MAX_INTERACTION_ORDER,
    round_nb=False,
)
cross_beta_rna = tc.add_bh_correction(glm.collect_terms(cross_glm_res_rna, interactions_only=False))
cross_inter_rna = glm.classify_interaction_terms(cross_beta_rna, term_col="term", gene_col="gene", unit_separator="|")
cross_inter_rna = glm.attach_motif_distance(cross_inter_rna, selected_seqlets)

# Accessibility, one fit per enhancer locus
cross_scram_effects_atac_long = ea.expand_locus_values(cross_scram_effects_raw, "ATAC_counts")
cross_glm_res_atac = glm.fit_all_loci(
    long_df=cross_scram_effects_atac_long,
    response_col="ATAC_counts",
    gene_col="name",
    locus_col="locus",
    family=GLM_FAMILY,
    max_order=MAX_INTERACTION_ORDER,
)
cross_beta_atac = glm.collect_terms(cross_glm_res_atac)
cross_inter_atac = glm.classify_interaction_terms(cross_beta_atac)

interdep = tc.add_bh_correction(glm.build_motif_interdependency(cross_beta_atac))
pair_agg_rna = glm.aggregate_pairs_to_enhancer(cross_inter_rna)
scope_rna = glm.summarise_scope_comparison(cross_inter_rna)

### Example loci

Seqlets of one gene's enhancers, and the GLM coefficients on expression and on
the accessibility of each enhancer.

In [ ]:
sidx = [(seq_idx, name) for seq_idx, name in enumerate(names) if "SLC25A37" in name.upper()]

fig, ax = plt.subplots(len(sidx), 1, figsize=(20, 9))
for a, (idx, name) in enumerate(sidx):
    ism_res = torch.tensor(ism_enhancer[names[idx]]["ATAC"]).permute(1, 0)
    plot_logo(ism_res, ax = ax[a],
          annotations=trimmed_seqlets.loc[trimmed_seqlets.seq_idx == idx, ["example_idx", "start","end", "attribution"]],
          score_key='attribution', n_tracks=5,)
    ax[a].set_ylabel(f"{name}", rotation = 90,)
plt.tight_layout()
plt.show()    

In [ ]:
for gene, loci in EXAMPLE_CROSS_LOCI.items():
    fig, ax = plt.subplots(1, 1 + len(loci), figsize=(26, 5))
    glmp.plot_coef_forest(cross_glm_res_rna[gene], ax=ax[0])
    for a, locus in zip(ax[1:], loci):
        glmp.plot_coef_forest(cross_glm_res_atac[f"{gene}@{locus}"], ax=a)
    plt.show()

### Within versus across enhancers

Distribution, significance and distance dependence of motif interactions by
scope, the recurrent factor-pair grammar, and the link to enhancer-level
interactions in silico and in the CRISPRi screen.

In [ ]:
cp.make_main_figure(cross_inter_rna, interdep, pair_agg=pair_agg_rna, enh_inters=enh_inters,
                    measured_pairs=measured_pairs.rename(columns={"interaction_beta": "coef"}))
plt.show()

In [ ]:
cp.make_grammar_figure(cross_inter_rna)
plt.show()

### Accessibility interdependency

Effect of single motifs and of motif pairs on the accessibility of every enhancer
of the gene. Pairs are *cis* (both motifs in the measured enhancer), *mixed* (one
inside) or *trans* (neither).

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(18, 18))
cp.plot_motif_interdependency(interdep, show_only_sig=True, hatch_nonsignificant=True,
                              ylabel_pad=53.0, gene_label_pad=54.0, ax=axes)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(6, 6))
cp.plot_pair_interdependency(interdep, ax=axes)
plt.show()

### Motif interactions and the enhancer pair

Aggregated across-enhancer motif interactions are compared with the enhancer-pair
coefficient in silico and with the screen. Because both are estimated per gene,
the correlation is also split into within-gene and between-gene parts.

In [ ]:
linked_pairs = cl.link_motif_pairs_to_enhancers(pair_agg_rna, enh_inters, measured_pairs)
pair_link = cl.summarise_pair_link(linked_pairs)

print("enhancer pairs:", len(linked_pairs),
      " with in silico coefficient:", linked_pairs["enh_coef"].notna().sum(),
      " with screen coefficient:", linked_pairs["interaction_beta"].notna().sum())
pair_link

In [ ]:
within_gene = cl.summarise_within_gene_link(linked_pairs, x_col="motif_mean", y_col="enh_coef", gene_col="gene")
within_gene

In [ ]:
within_gene_measured = cl.summarise_within_gene_link(linked_pairs, x_col="motif_mean", y_col="interaction_beta", gene_col="gene")
within_gene_measured

### Text summary

All cross-enhancer statistics in `results/summaries/cross_enhancer_summary.txt`.

In [ ]:
cross_summary = cr.write_cross_summary(
    paths.SUMMARIES / "cross_enhancer_summary.txt",
    inters_rna=cross_inter_rna,
    beta_rna=cross_beta_rna,
    inters_atac=cross_inter_atac,
    beta_atac=cross_beta_atac,
    interdep=interdep,
    pair_agg=pair_agg_rna,
    enh_inters=enh_inters,
    enh_mains=enh_mains,
    tf_pairs=tc.add_bh_correction(tc.build_pairwise_interactions(scram_effects_raw, "ATAC_counts", seqlets=fulls_seqlets)),
    glm_results_rna=cross_glm_res_rna,
    glm_results_atac=cross_glm_res_atac,
    top_n=15,
    exemplar_n=10,
)